# Multi-Region Iteration Selector and CSV Export

This notebook:

1. Loads inversion results for `NW`, `NE`, `SE`, `SW`, and `tielines`.
2. Lets you choose a different iteration for each region.
3. Shows a combined resistivity map (left) and line-by-line data comparison plot (right).
4. Exports one merged CSV using the currently selected iterations.

> Expected inversion files: `data/{area}_inv_results_atem_full.pik`

In [ ]:
import os
import sys
from pathlib import Path

import dill
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib.colors import LogNorm
from ipywidgets import widgets, VBox, HBox, interactive_output
from simpeg.electromagnetics.utils.em1d_utils import get_vertical_discretization

# Project rhttps://ostrnrcan-dostrncan.canada.caeoot hanedling so the notebook works when opened from notebooks/.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

if "tools" not in sys.path:
    sys.path.append("tools")

from tools import binning

AREAS = ["NW", "NE", "SE", "SW", "tielines"]
DX_BIN = 50

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 12

print(f"cwd: {os.getcwd()}")

cwd: /Volumes/X31/01.Projects/atem


## 1) Load and Merge Inversion Results from All Regions

Load each region's binned data and inversion dictionary, then normalize into a shared structure.

In [7]:
def build_region_payload(area: str, dx: int = 50) -> dict:
    tmp = binning(dx=dx, area=area)
    out_path = Path("data") / f"{area}_inv_results_atem_full.pik"
    if not out_path.exists():
        raise FileNotFoundError(f"Missing inversion file: {out_path}")

    out_dict = dill.load(open(out_path, "rb"))

    values = tmp["values"]
    values_std = tmp["values_std"]
    soundings = tmp["soundings"]
    dheader = tmp["dheader"]
    picker = tmp["picker"]

    df_binned = pd.DataFrame(np.vstack(values), columns=["Line", "distance"] + picker[1:])
    df_std = pd.DataFrame(np.vstack(values_std), columns=["bheight"] + dheader)

    n_sounding = len(df_binned)
    n_time = len(dheader)
    iterations = len(out_dict.keys())

    # Infer number of layers from model length and sounding count.
    n_layer = int(np.asarray(out_dict[1]["m"]).size / n_sounding)

    thickness = get_vertical_discretization(21, 2, 1.17)
    if len(thickness) + 1 != n_layer:
        # Fallback when inversion used a different layer count.
        thickness = np.r_[np.ones(n_layer - 1), 1.0]

    hz = np.r_[thickness, thickness[-1]]
    depth_edges = np.r_[0.0, np.cumsum(hz)]
    depth_centers = 0.5 * (depth_edges[:-1] + depth_edges[1:])

    dobs = df_binned[dheader].values.astype(float)

    line_no = list(df_binned["Line"].astype(str).unique())
    distance_by_line = {}
    line_start_end = {}
    start = 0
    for i_line, line_name in enumerate(line_no):
        n_in_line = int(soundings[i_line])
        end = start + n_in_line
        distance_by_line[line_name] = df_binned.iloc[start:end]["distance"].astype(float).to_numpy()
        line_start_end[line_name] = (start, end)
        start = end

    # Use active data count (same logic as inversion notebook) for normalized misfit.
    data_vals = df_binned[dheader].values.astype(float)
    data_std = df_std[dheader].values.astype(float)
    channel_id = np.tile(np.arange(data_vals.shape[1]), (data_vals.shape[0], 1))
    with np.errstate(divide="ignore", invalid="ignore"):
        data_rerr = data_std / np.abs(data_vals)
    criteria_rerr = 0.03
    cut_off = (data_rerr > criteria_rerr) * (channel_id >= 0)
    n_total_data = int(data_vals.size)
    n_active_data = int(n_total_data - np.nansum(cut_off))
    if n_active_data <= 0:
        raise ValueError(f"{area}: computed active data count is {n_active_data}")

    phi_d = np.array([out_dict[i]["phi_d"] for i in range(1, iterations + 1)], dtype=float)
    chi2 = phi_d / n_active_data

    return {
        "area": area,
        "tmp": tmp,
        "outDict": out_dict,
        "df_data_binned": df_binned,
        "df_data_std_binned": df_std,
        "dheader": dheader,
        "soundings": soundings,
        "line_no": line_no,
        "line_start_end": line_start_end,
        "distance_by_line": distance_by_line,
        "dobs": dobs,
        "n_sounding": n_sounding,
        "n_time": n_time,
        "n_layer": n_layer,
        "iterations": iterations,
        "depth_edges": depth_edges,
        "depth_centers": depth_centers,
        "n_total_data": n_total_data,
        "n_active_data": n_active_data,
        "chi2": chi2,
    }


region_data = {}
for area in AREAS:
    region_data[area] = build_region_payload(area, dx=DX_BIN)

summary_df = pd.DataFrame(
    {
        "area": AREAS,
        "n_soundings": [region_data[a]["n_sounding"] for a in AREAS],
        "n_lines": [len(region_data[a]["line_no"]) for a in AREAS],
        "n_layers": [region_data[a]["n_layer"] for a in AREAS],
        "n_iterations": [region_data[a]["iterations"] for a in AREAS],
        "n_total_data": [region_data[a]["n_total_data"] for a in AREAS],
        "n_active_data": [region_data[a]["n_active_data"] for a in AREAS],
    }
)
summary_df

>> Depth from the surface to the base of the bottom layer is 306.3m
>> Depth from the surface to the base of the bottom layer is 306.3m
>> Depth from the surface to the base of the bottom layer is 306.3m
>> Depth from the surface to the base of the bottom layer is 306.3m
>> Depth from the surface to the base of the bottom layer is 306.3m


,area,n_soundings,n_lines,n_layers,n_iterations,n_total_data,n_active_data
0,NW,98552,118,22,20,2168144,1399778
1,NE,134179,136,22,20,2951938,2080562
2,SE,105135,87,22,20,2312970,1818497
3,SW,170440,115,22,20,3749680,2745380
4,tielines,53169,57,22,20,1169718,787516


## 2) Visualize Best Iteration Selection with Interactive Sliders

Use one slider per region, then inspect per-area misfit trend ($\chi^2$) to choose best iterations.

In [8]:
iter_sliders = {
    area: widgets.IntSlider(
        min=1,
        max=region_data[area]["iterations"],
        step=1,
        value=region_data[area]["iterations"],
        description=f"{area} iter",
        continuous_update=False,
        style={"description_width": "70px"},
        layout=widgets.Layout(width="320px"),
    )
    for area in AREAS
}

selected_region_widget = widgets.Dropdown(
    options=AREAS,
    value=AREAS[0],
    description="Plot area",
)

line_widget = widgets.Dropdown(description="Line")

def _update_line_options(*_):
    area = selected_region_widget.value
    lines = region_data[area]["line_no"]
    line_widget.options = lines
    if len(lines) > 0:
        line_widget.value = lines[0]

_update_line_options()
selected_region_widget.observe(_update_line_options, names="value")

common_n_layer = min(region_data[a]["n_layer"] for a in AREAS)
depth_widget = widgets.IntSlider(
    min=0,
    max=common_n_layer - 1,
    step=1,
    value=0,
    description="Depth idx",
    continuous_update=False,
    style={"description_width": "70px"},
    layout=widgets.Layout(width="320px"),
)


def selected_iterations() -> dict:
    return {area: int(iter_sliders[area].value) for area in AREAS}


def plot_misfit_by_area(_=None):
    fig, axes = plt.subplots(2, 3, figsize=(16, 8), constrained_layout=True)
    axes = axes.ravel()

    current = selected_iterations()
    for i, area in enumerate(AREAS):
        ax = axes[i]
        chi2 = region_data[area]["chi2"]
        it = np.arange(1, len(chi2) + 1)
        ax.plot(it, chi2, "o-", lw=1.8, ms=4, color="tab:blue")
        ax.axhline(1.0, ls="--", c="gray", lw=1.0)
        ax.axvline(current[area], ls="--", c="tab:red", lw=1.2)
        ax.set_yscale("log")
        ax.set_title(f"{area} | selected={current[area]}")
        ax.set_xlabel("Iteration")
        ax.set_ylabel("chi2")
        ax.grid(alpha=0.25)

    axes[-1].axis("off")
    plt.show()


misfit_btn = widgets.Button(description="Update misfit plots", button_style="info")
misfit_out = widgets.Output()


def _on_misfit_click(_):
    with misfit_out:
        misfit_out.clear_output(wait=True)
        plot_misfit_by_area()


misfit_btn.on_click(_on_misfit_click)

iter_rows = [HBox([iter_sliders[a]]) for a in AREAS]
ui_iter = VBox(iter_rows + [misfit_btn, misfit_out])
ui_iter

## 3) Display Merged Inversion Results and Data Comparison Plots

Left: merged resistivity map at selected depth using each region's chosen iteration.

Right: observed vs predicted response by line for selected region.

In [9]:
def _resistivity_at_depth(area: str, i_iteration: int, i_depth: int):
    payload = region_data[area]
    m_iter = np.asarray(payload["outDict"][i_iteration]["m"], dtype=float)
    rho = 1.0 / np.exp(m_iter)
    rho_layers = rho.reshape((payload["n_sounding"], payload["n_layer"]))
    return rho_layers[:, i_depth]


def plot_merged_and_line(
    depth_idx,
    plot_area,
    line_name,
    NW_iter,
    NE_iter,
    SE_iter,
    SW_iter,
    tielines_iter,
):
    iter_map = {
        "NW": int(NW_iter),
        "NE": int(NE_iter),
        "SE": int(SE_iter),
        "SW": int(SW_iter),
        "tielines": int(tielines_iter),
    }

    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(18, 6), constrained_layout=True)

    # Left panel: merged resistivity depth slice from all regions.
    cmap = "turbo"
    norm = LogNorm(vmin=8, vmax=60)
    for area in AREAS:
        payload = region_data[area]
        i_depth = int(np.clip(depth_idx, 0, payload["n_layer"] - 1))
        rho_depth = _resistivity_at_depth(area, iter_map[area], i_depth)

        x = pd.to_numeric(payload["df_data_binned"]["x_wgs84"], errors="coerce").to_numpy(dtype=float)
        y = pd.to_numeric(payload["df_data_binned"]["y_wgs84"], errors="coerce").to_numpy(dtype=float)

        valid = np.isfinite(x) & np.isfinite(y) & np.isfinite(rho_depth)
        sc = ax_left.scatter(
            x[valid],
            y[valid],
            c=rho_depth[valid],
            s=4,
            cmap=cmap,
            norm=norm,
            alpha=0.75,
            label=f"{area} (iter {iter_map[area]})",
            linewidths=0,
        )

    cbar = plt.colorbar(sc, ax=ax_left, fraction=0.046, pad=0.02)
    cbar.set_label("Resistivity (ohm-m)")
    ax_left.set_title(f"Merged resistivity map | depth idx={depth_idx}")
    ax_left.set_xlabel("x_wgs84")
    ax_left.set_ylabel("y_wgs84")
    ax_left.set_aspect("equal", adjustable="box")
    ax_left.grid(alpha=0.15)
    ax_left.legend(loc="best", fontsize=8)

    # Right panel: selected region line-by-line data comparison.
    payload = region_data[plot_area]
    i_iteration = iter_map[plot_area]
    n_time = payload["n_time"]

    if str(line_name) not in payload["line_start_end"]:
        ax_right.set_title(f"No line '{line_name}' in {plot_area}")
        ax_right.axis("off")
        plt.show()
        return

    start, end = payload["line_start_end"][str(line_name)]
    dist = payload["distance_by_line"][str(line_name)]
    sort_idx = np.argsort(dist)
    dist_sorted = dist[sort_idx]

    dpred = np.asarray(payload["outDict"][i_iteration]["dpred"], dtype=float).reshape((-1, n_time))
    dobs = payload["dobs"]

    dpred_line = -dpred[start:end, :][sort_idx, :]
    dobs_line = -dobs[start:end, :][sort_idx, :]

    for i_time in range(n_time):
        obs_label = "obs (channels)" if i_time == 0 else None
        pred_label = "pred (channels)" if i_time == 0 else None
        ax_right.plot(dist_sorted, np.abs(dobs_line[:, i_time]), "k--", lw=0.7, alpha=0.25, label=obs_label)
        ax_right.plot(dist_sorted, np.abs(dpred_line[:, i_time]), color="tab:blue", lw=0.8, alpha=0.25, label=pred_label)

    ax_right.set_yscale("log")
    ax_right.set_xlabel("Distance along line (m)")
    ax_right.set_ylabel("|dB/dt|")
    ax_right.set_title(f"{plot_area} | line={line_name} | iter={i_iteration}")
    ax_right.grid(alpha=0.2)
    ax_right.legend()

    plt.show()


controls_box = VBox(
    [
        HBox([selected_region_widget, line_widget, depth_widget]),
        HBox([iter_sliders[a] for a in AREAS]),
    ]
)

plot_out = interactive_output(
    plot_merged_and_line,
    {
        "depth_idx": depth_widget,
        "plot_area": selected_region_widget,
        "line_name": line_widget,
        "NW_iter": iter_sliders["NW"],
        "NE_iter": iter_sliders["NE"],
        "SE_iter": iter_sliders["SE"],
        "SW_iter": iter_sliders["SW"],
        "tielines_iter": iter_sliders["tielines"],
    },
)

VBox([controls_box, plot_out])

## 4) Export Selected Iterations to CSV

Export one merged table using current slider values.

In [10]:
def build_export_dataframe(iter_map: dict) -> pd.DataFrame:
    frames = []

    for area in AREAS:
        payload = region_data[area]
        i_iteration = int(iter_map[area])

        m_iter = np.asarray(payload["outDict"][i_iteration]["m"], dtype=float)
        rho = 1.0 / np.exp(m_iter)
        rho_layers = rho.reshape((payload["n_sounding"], payload["n_layer"]))

        keep_cols = [
            c
            for c in ["Line", "distance", "x_wgs84", "y_wgs84", "dtm", "bheight", "pwrline", "TranPeak"]
            if c in payload["df_data_binned"].columns
        ]
        df_base = payload["df_data_binned"][keep_cols].copy()
        df_base["region"] = area
        df_base["selected_iteration"] = i_iteration
        df_base["sounding_index_in_region"] = np.arange(payload["n_sounding"], dtype=int)

        rho_cols = [f"rho_layer_{i + 1:02d}" for i in range(payload["n_layer"])]
        df_rho = pd.DataFrame(rho_layers, columns=rho_cols)

        frames.append(pd.concat([df_base.reset_index(drop=True), df_rho], axis=1))

    out = pd.concat(frames, ignore_index=True)
    return out


def export_selected_iterations(out_dir: str = "data") -> Path:
    iters = selected_iterations()
    df = build_export_dataframe(iters)

    fname = "merged_inversion_selected_" + "_".join([f"{a}{iters[a]}" for a in AREAS]) + ".csv"
    out_path = Path(out_dir) / fname
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)

    print(f"Saved: {out_path}")
    print(f"Rows: {len(df):,}, Cols: {len(df.columns):,}")
    return out_path


export_btn = widgets.Button(description="Export merged CSV", button_style="success")
export_out = widgets.Output()


def _on_export_click(_):
    with export_out:
        export_out.clear_output(wait=True)
        export_selected_iterations("data")


export_btn.on_click(_on_export_click)
VBox([export_btn, export_out])

In [6]:
# Optional helper: apply explicit iteration list in AREA order.
def set_iterations(values):
    if len(values) != len(AREAS):
        raise ValueError(f"Need {len(AREAS)} values in this order: {AREAS}")
    for area, val in zip(AREAS, values):
        val = int(val)
        max_it = region_data[area]["iterations"]
        if val < 1 or val > max_it:
            raise ValueError(f"{area}: iteration {val} out of range [1, {max_it}]")
        iter_sliders[area].value = val


# Example requested in the prompt
set_iterations([11, 15, 18, 19, 10])
selected_iterations()

{'NW': 11, 'NE': 15, 'SE': 18, 'SW': 19, 'tielines': 10}

In [ ]:
# Optional direct one-liner export without clicking the button:
# export_selected_iterations("data")

## 5) Validate Exported CSV by Plotting Resistivity Section

Load the exported merged CSV and draw a terrain-following resistivity section for a selected region and line.

In [11]:
from pathlib import Path
from matplotlib.colors import LogNorm


def _cell_edges_1d(values):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return np.array([], dtype=float)
    if values.size == 1:
        v = values[0]
        return np.array([v - 0.5, v + 0.5], dtype=float)
    mids = 0.5 * (values[:-1] + values[1:])
    left = values[0] - (mids[0] - values[0])
    right = values[-1] + (values[-1] - mids[-1])
    return np.r_[left, mids, right]


def _latest_export_csv(data_dir="data"):
    candidates = sorted(Path(data_dir).glob("merged_inversion_selected_*.csv"), key=lambda p: p.stat().st_mtime)
    if candidates:
        return candidates[-1]

    # Fallback: build one immediately from current region_data using last iteration per area.
    iter_map = {a: int(region_data[a]["iterations"]) for a in AREAS}
    df_fallback = build_export_dataframe(iter_map)
    out_path = Path(data_dir) / "merged_inversion_selected_autogen.csv"
    df_fallback.to_csv(out_path, index=False)
    return out_path


csv_path = _latest_export_csv("data")
df_export = pd.read_csv(csv_path)

rho_cols = [c for c in df_export.columns if c.startswith("rho_layer_")]
if len(rho_cols) == 0:
    raise ValueError(f"No resistivity columns found in {csv_path}")

print(f"Loaded CSV: {csv_path}")
print(f"Rows: {len(df_export):,}, Layers: {len(rho_cols)}")

region_csv_widget = widgets.Dropdown(
    options=sorted(df_export["region"].dropna().unique().tolist()),
    description="Region",
)
line_csv_widget = widgets.Dropdown(description="Line")


def _update_line_csv_options(*_):
    reg = region_csv_widget.value
    df_r = df_export[df_export["region"].astype(str) == str(reg)]
    lines = sorted(df_r["Line"].astype(str).unique().tolist())
    line_csv_widget.options = lines
    if lines:
        line_csv_widget.value = lines[0]


_update_line_csv_options()
region_csv_widget.observe(_update_line_csv_options, names="value")


iteration_csv_widget = widgets.IntText(value=-1, description="Iter", disabled=True)


def plot_section_from_csv(region, line_name):
    df_line = df_export[
        (df_export["region"].astype(str) == str(region))
        & (df_export["Line"].astype(str) == str(line_name))
    ].copy()

    if df_line.empty:
        print(f"No rows for region={region}, line={line_name}")
        return

    df_line["distance"] = pd.to_numeric(df_line["distance"], errors="coerce")
    df_line["dtm"] = pd.to_numeric(df_line["dtm"], errors="coerce")
    df_line = df_line.sort_values("distance").reset_index(drop=True)

    dist = df_line["distance"].to_numpy(dtype=float)
    dtm = df_line["dtm"].to_numpy(dtype=float)

    rho = df_line[rho_cols].to_numpy(dtype=float)
    n_layer = rho.shape[1]

    # Use region depth edges from original payload to reconstruct terrain-following mesh.
    depth_edges = np.asarray(region_data[str(region)]["depth_edges"], dtype=float)
    if depth_edges.size != (n_layer + 1):
        # Fallback to unit spacing if CSV layer count differs unexpectedly.
        depth_edges = np.arange(n_layer + 1, dtype=float)

    x_edges = _cell_edges_1d(dist)
    dtm_edges = _cell_edges_1d(dtm)

    X_edges = np.tile(x_edges[np.newaxis, :], (n_layer + 1, 1))
    Z_edges = dtm_edges[np.newaxis, :] - depth_edges[:, np.newaxis]

    section = rho.T

    fig, ax = plt.subplots(figsize=(14, 4), constrained_layout=True)
    pcm = ax.pcolormesh(
        X_edges,
        Z_edges,
        section,
        cmap="turbo",
        norm=LogNorm(vmin=8, vmax=60),
        shading="auto",
    )
    cbar = plt.colorbar(pcm, ax=ax, orientation="horizontal", pad=0.12, fraction=0.08)
    cbar.set_label("Resistivity (ohm-m)")

    sel_iter = df_line["selected_iteration"].dropna().unique()
    iter_label = int(sel_iter[0]) if len(sel_iter) == 1 else "mixed"
    iteration_csv_widget.value = -1 if iter_label == "mixed" else iter_label

    ax.set_title(f"CSV validation section | {region} | line={line_name} | iter={iter_label}")
    ax.set_xlabel("Distance along line (m)")
    ax.set_ylabel("Elevation (m)")
    ax.grid(False)
    plt.show()


csv_plot_out = interactive_output(
    plot_section_from_csv,
    {
        "region": region_csv_widget,
        "line_name": line_csv_widget,
    },
)

VBox([
    HBox([region_csv_widget, line_csv_widget]),
    csv_plot_out,
])

Loaded CSV: data/merged_inversion_selected_NW20_NE15_SE16_SW20_tielines20.csv
Rows: 561,475, Layers: 22
